# Gemma-4 held-out evaluation (ARCHITECTURE.md §2.6, §8.1)

Scores a policy on the frozen holdout. Two runs per session: the SFT adapter
(`checkpoints/sft`) and the zero-shot base, on the *same* task slice, so
resolved-rate and patch-validity are directly comparable.

Full 291-task x 2-policy eval is ~100+ GPU-hours at the measured T4 speed,
so the committed `data/holdout_eval_subset.jsonl` (24 tasks, stratified
round-robin by repo) is the headline evaluation; the limit is disclosed.

The subset is also cleaned at the commit/issue level: the 23 holdout tasks
that share a base commit or issue text with the SFT/RL pools (AUDIT M5) are
excluded before sampling.

Set `EVAL_LIMIT=10` (default) for a smoke run; `EVAL_LIMIT=0` for the full
holdout. Sandbox isolation on Kaggle is `best_effort` (unshare is
seccomp-blocked); the script warns and the numbers are disclosed as such.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys
import zipfile

assert os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_GPU_RUNNING_TYPE"), (
    "Turn on the GPU runtime first (T4/P100) — ARCHITECTURE §5"
)

repo_dir = pathlib.Path("gemma4-coding-agent")

def _extract(root: pathlib.Path) -> bool:
    zips = sorted(root.rglob("gemma4-coding-agent.zip"))
    if zips:
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(".")
        if repo_dir.is_dir():
            return True
    nested = root / "gemma4-coding-agent"
    if nested.is_dir() and (nested / "pyproject.toml").exists():
        if not repo_dir.exists():
            shutil.copytree(nested, repo_dir)
        return True
    if (root / "pyproject.toml").exists() and (root / "ARCHITECTURE.md").exists():
        if not repo_dir.exists():
            shutil.copytree(root, repo_dir, ignore=shutil.ignore_patterns("*.zip", "*.tar"))
        return True
    return False

# Fresh public clone FIRST: the attached dataset snapshot can go stale.
ok = False
try:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ashura-asura/gemma4-coding-agent.git"],
                   check=True, timeout=180)
    ok = repo_dir.is_dir() and (repo_dir / "pyproject.toml").exists()
except Exception as _e:
    print("fresh clone failed, falling back to the attached dataset:", _e)
if not ok:
    if repo_dir.exists():
        shutil.rmtree(repo_dir, ignore_errors=True)
    inputs = sorted(pathlib.Path("/kaggle/input").glob("*gemma4*"))
    ok = any(_extract(p) for p in inputs)
if not ok:
    import kagglehub
    dl = pathlib.Path(kagglehub.dataset_download("ashuraasura/gemma4-coding-agent-repo"))
    ok = _extract(dl)
assert ok, f"repo bootstrap failed; /kaggle/input = {sorted(p.name for p in pathlib.Path('/kaggle/input').iterdir())}"
os.chdir(repo_dir.resolve())
for _sh in pathlib.Path("scripts").glob("*.sh"):
    _sh.write_bytes(_sh.read_bytes().replace(b"\r\n", b"\n"))
subprocess.run(f'"{sys.executable}" -m pip install -q -e ".[train]"', shell=True, check=True)
import hashlib as _hf
_commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "no-git (dataset snapshot)"
print("repo ready at", pathlib.Path.cwd(), "| commit:", _commit,
      "| run_eval sha:", _hf.sha1(pathlib.Path("eval/run_eval.py").read_bytes()).hexdigest()[:12])


In [ ]:
# pre-flight (CPU): harness tests + dry-run task listing
!python -m pytest eval/tests -q
!python -m eval.run_eval --config configs/eval_config.yaml --dry-run


In [ ]:
# SFT adapter input: Kaggle auto-extracts uploaded zip datasets, so accept both layouts.
import glob

dest = pathlib.Path("checkpoints/sft")
dest.mkdir(parents=True, exist_ok=True)
zips = glob.glob("/kaggle/input/**/sft_adapter.zip", recursive=True)
if zips:
    with zipfile.ZipFile(zips[0]) as zf:
        print("zip contains:", zf.namelist()[:3])
        for name in zf.namelist():
            rel = name
            if rel.startswith("checkpoints/sft/"):
                rel = rel[len("checkpoints/sft/"):]
            elif rel.startswith("sft/"):
                rel = rel[len("sft/"):]
            if not rel or name.endswith("/"):
                continue
            target = dest / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(zf.read(name))
else:
    roots = [p.parent for p in pathlib.Path("/kaggle/input").rglob("adapter_config.json")]
    assert roots, "no sft_adapter.zip and no adapter_config.json under /kaggle/input"
    src = roots[0]
    print("dataset-extracted adapter at", src)
    for item in src.iterdir():
        target = dest / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)
assert (dest / "adapter_config.json").exists(), "adapter_config.json missing after import"
print("adapter at", sorted(p.name for p in dest.iterdir())[:8])


In [ ]:
# Smoke (EVAL_LIMIT=6) or the full documented subset (EVAL_LIMIT=0), adapter then base.
# Full 291-task holdout is ~100+ GPU-hours; the committed subset is stratified
# round-robin by repo (24 tasks) and is the headline eval (see README).
import os
import subprocess
import sys
import time

os.environ.setdefault("SANDBOX_FS_ISOLATION", "best_effort")  # Kaggle blocks unshare (probe-verified)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
LIMIT = os.environ.get("EVAL_LIMIT", "0")
TASKS = os.environ.get("EVAL_TASKS", "data/holdout_eval_subset.jsonl")
limit_args = [] if LIMIT in ("", "0", "all") else ["--limit", LIMIT]
base = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"
runs = [("sft", ["--policy", "hf", "--model", "checkpoints/sft"]),
        ("base", ["--policy", "hf", "--model", base])]
pathlib.Path("eval/results").mkdir(parents=True, exist_ok=True)
for name, extra in runs:
    out = f"eval/results/{name}.jsonl"
    cmd = [sys.executable, "-m", "eval.run_eval", "--config", "configs/eval_config.yaml",
           "--tasks", TASKS, "--max-steps", "12", "--max-new-tokens", "512",
           "--temperature", "0",  # greedy: one fixed outcome per task for McNemar
           "--quantize", *limit_args, "--out", out, "--dump-trajectories", f"eval/results/{name}_traj.jsonl", *extra]
    print("RUN", " ".join(cmd), flush=True)
    started = time.time()
    try:
        subprocess.run(cmd, check=True)
        subprocess.run([sys.executable, "-m", "eval.report", out, "--out", f"eval/results/{name}_summary.json"], check=True)
    except subprocess.CalledProcessError as exc:
        # A hard failure in one arm must not discard the other arm's results.
        print(f"RUN {name} failed with exit {exc.returncode}; continuing to next arm", flush=True)
    print(f"RUN {name} elapsed {time.time() - started:.0f}s", flush=True)

if pathlib.Path("eval/results/sft.jsonl").exists() and pathlib.Path("eval/results/base.jsonl").exists():
    subprocess.run([sys.executable, "-m", "eval.report", "eval/results/sft.jsonl",
                    "--baseline", "eval/results/base.jsonl"], check=False)


In [ ]:
import subprocess

!du -sh eval/results
subprocess.run(f'"{sys.executable}" -m zipfile -c /kaggle/working/eval_results.zip eval/results', shell=True, check=True)
print("download eval_results.zip from /kaggle/working")
